# Notebook 3 — Adaptive Reinforcement Learning Fusion (CNN + ViT)

This notebook adaptively fuses predictions from the **CNN branch** (`01_cnn_branch.ipynb`) and the **ViT branch** (`02_vit_branch.ipynb`).

### Reinforcement Learning Formulation
- **State ($s$)**: Class context (true label $y$ during calibration; proxy estimate $\hat{y}_0$ during inference).
- **Action ($a$ / $w$ )**: Continuous fusion weight $w_{\text{cnn}} \in [w_{\min}, w_{\max}]$ allocated to the CNN (with $w_{\text{vit}} = 1 - w_{\text{cnn}}$).
- **Reward ($r$)**: Bounded per-model confidence signal $r_m = 2 \cdot p_m(y_{\text{true}}) - 1 \in [-1, 1]$ directly derived from the softmax probability assigned to the ground-truth class.

---

## 1. Data-Split Discipline (Non-Negotiable)
> **Crucial Reviewer Requirement:**
> All fusion weights and confidence parameters are calibrated strictly on the **VAL exports** (`cnn_val_export.pt` and `vit_val_export.pt`).
>
> The **TEST exports** (`cnn_test_export.pt` and `vit_test_export.pt`) are loaded and evaluated **exactly once** at the very end of this notebook, purely to measure and report the final unbiased generalization accuracy.
>
> Reusing test data to tune or calibrate fusion weights invalidates experimental conclusions and constitutes data leakage.

---

## 2. Algorithm Rationale & Rejected Alternatives

Our calibration algorithm employs **per-class TD(0) sample-mean advantage**, **UCB-style confidence shrinkage**, and a **sigmoid saturation link**.

Before finalizing this scheme, two standard alternatives were evaluated in simulation and rejected for concrete mathematical reasons:

### Why Classic Multi-Armed Bandit UCB Was Rejected
Classic UCB (Upper Confidence Bound) arm-selection uses an exploration bonus specifically designed for **partial-feedback (bandit) settings**, where the agent only observes the reward of the chosen arm.
In our fusion setting, both models evaluate every sample, and their correctness/probabilities against ground truth are **simultaneously observed** on every calibration instance. This is a **full-information setting**; an arm-selection exploration bonus has nothing to explore.
However, UCB's fundamental insight — that estimation uncertainty shrinks as $\mathcal{O}(1 / \sqrt{n})$ — is retained as a **confidence-shrinkage factor** to regularize classes with few calibration samples back toward the unbiased prior ($w = 0.5$).

### Why Constant-Learning-Rate Multiplicative Weights / Hedge Was Rejected
A standard multiplicative weights update ($w_i \leftarrow w_i \cdot e^{\eta r_i}$, normalized) with constant learning rate $\eta$ is a **martingale in log-odds space with no mean reversion**.
Under simulation with two models of identical true skill, constant-LR multiplicative weights does **not** stay at $0.5$; it undergoes an unconstrained random walk and drifts to extreme weights (near $0$ or $1$) purely from noise, with no restoring force.
Because both models are frozen and fully trained, their true per-class skill is **stationary**.
Under the **Robbins-Monro condition** ($\sum \alpha_n = \infty$, $\sum \alpha_n^2 < \infty$), a decreasing step size $\alpha_n = 1/n$ (the exact running sample mean) provably converges to the stationary expected advantage.

### Why Sigmoid / Tanh Instead of Log(x)
Raw $\log(x)$ is often proposed as a diminishing-returns function, but $\log(x)$ is mathematically **unbounded** as $x \to \infty$. To satisfy the hard requirement that fusion weights never exceed preset safe bounds ($w_{\min}=0.05, w_{\max}=0.95$), a **sigmoid** function is employed because it possesses true, finite horizontal asymptotes.

In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set seeds for exact reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Core Algorithm Constants
NUM_CLASSES = 10
k = 2.0             # Sigmoid steepness
n0 = 30             # Confidence warm-up sample count
w_min = 0.05        # Minimum weight bound (prevents total reliance on one model)
w_max = 0.95        # Maximum weight bound
CONF_THRESHOLD = 0.3 # Minimum calibration confidence required before falling back to global weight

print(f"Fusion Hyperparameters: k={k}, n0={n0}, w_min={w_min}, w_max={w_max}, fallback_thresh={CONF_THRESHOLD}")

Fusion Hyperparameters: k=2.0, n0=30, w_min=0.05, w_max=0.95, fallback_thresh=0.3


## 3. Load Branch Exports (Val & Test)

We load `cnn_val_export.pt`, `cnn_test_export.pt`, `vit_val_export.pt`, and `vit_test_export.pt`.
We verify that shapes, labels, and contracts match between both branches.

In [2]:
# Load CNN branch exports
cnn_val = torch.load("cnn_val_export.pt", weights_only=False)
cnn_test = torch.load("cnn_test_export.pt", weights_only=False)

# Load ViT branch exports
vit_val = torch.load("vit_val_export.pt", weights_only=False)
vit_test = torch.load("vit_test_export.pt", weights_only=False)

# Verify label alignment
assert torch.equal(cnn_val["label"], vit_val["label"]), "VAL ground-truth labels do not match!"
assert torch.equal(cnn_test["label"], vit_test["label"]), "TEST ground-truth labels do not match!"

print("Loaded Exports Successfully:")
print(f"  Val CNN Acc:  {cnn_val['correct'].float().mean().item():.4f} | Val ViT Acc:  {vit_val['correct'].float().mean().item():.4f}")
print(f"  Test CNN Acc: {cnn_test['correct'].float().mean().item():.4f} | Test ViT Acc: {vit_test['correct'].float().mean().item():.4f}")

Loaded Exports Successfully:
  Val CNN Acc:  0.7625 | Val ViT Acc:  0.9737
  Test CNN Acc: 0.7875 | Test ViT Acc: 0.9700


## 4. Fusion Algorithm Class & Initialization Invariant Unit Test

### Initialization Invariant (Hard Requirement)
Before any calibration sample of class $s$ has been observed ($n[s] = 0$):
- $\text{confidence} = \min(1, 0 / n_0) = 0$
- $\text{advantage} \cdot \text{confidence} = 0$
- $\sigma(0) = 0.5$
- $w_{\text{cnn}}[s] = w_{\min} + (w_{\max} - w_{\min}) \cdot 0.5 = 0.05 + 0.90 \cdot 0.5 = 0.50$
- $w_{\text{vit}}[s] = 1 - w_{\text{cnn}}[s] = 0.50$

This guarantees that the system begins in a completely unbiased, symmetric state. The unit test below explicitly asserts this contract.

In [3]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

class RLFusionCalibrator:
    def __init__(self, num_classes=NUM_CLASSES, k=k, n0=n0, w_min=w_min, w_max=w_max):
        self.num_classes = num_classes
        self.k = k
        self.n0 = n0
        self.w_min = w_min
        self.w_max = w_max
        self.reset()

    def reset(self):
        self.n = np.zeros(self.num_classes, dtype=np.int64)
        self.Q_cnn = np.zeros(self.num_classes, dtype=np.float64)
        self.Q_vit = np.zeros(self.num_classes, dtype=np.float64)

        # Global pooled state (single state across all classes for ablation and fallback)
        self.n_global = 0
        self.Q_cnn_global = 0.0
        self.Q_vit_global = 0.0

    def get_confidence(self, s):
        return min(1.0, self.n[s] / self.n0)

    def get_global_confidence(self):
        return min(1.0, self.n_global / self.n0)

    def compute_weights(self, advantage, confidence):
        w_c = self.w_min + (self.w_max - self.w_min) * sigmoid(self.k * advantage * confidence)
        w_v = 1.0 - w_c
        return float(w_c), float(w_v)

    def get_class_weights(self, s):
        adv = self.Q_cnn[s] - self.Q_vit[s]
        conf = self.get_confidence(s)
        return self.compute_weights(adv, conf)

    def get_global_weights(self):
        adv = self.Q_cnn_global - self.Q_vit_global
        conf = self.get_global_confidence()
        return self.compute_weights(adv, conf)

    def update(self, s, r_cnn, r_vit):
        # 1. Update per-class statistics
        self.n[s] += 1
        self.Q_cnn[s] += (1.0 / self.n[s]) * (r_cnn - self.Q_cnn[s])
        self.Q_vit[s] += (1.0 / self.n[s]) * (r_vit - self.Q_vit[s])

        # 2. Update global pooled statistics (for single-state ablation / fallback)
        self.n_global += 1
        self.Q_cnn_global += (1.0 / self.n_global) * (r_cnn - self.Q_cnn_global)
        self.Q_vit_global += (1.0 / self.n_global) * (r_vit - self.Q_vit_global)

# --- UNIT TEST: Initialization Invariant Contract ---
fresh_calibrator = RLFusionCalibrator()
for c in range(NUM_CLASSES):
    w_c, w_v = fresh_calibrator.get_class_weights(c)
    assert np.isclose(w_c, 0.5), f"Unit Test Failed: class {c} initial w_cnn is {w_c}, expected 0.5"
    assert np.isclose(w_v, 0.5), f"Unit Test Failed: class {c} initial w_vit is {w_v}, expected 0.5"

w_cg, w_vg = fresh_calibrator.get_global_weights()
assert np.isclose(w_cg, 0.5) and np.isclose(w_vg, 0.5), "Unit Test Failed on global weight initialization!"
print("[PASS] Initialization Invariant Unit Test PASSED: All unvisited states start at exactly (0.50, 0.50).")

[PASS] Initialization Invariant Unit Test PASSED: All unvisited states start at exactly (0.50, 0.50).


## 5. Calibration Loop on VAL Split

We iterate through all samples in the **VAL** split in sequential order:
1. State $s = y$ (true ground-truth class is known during calibration).
2. Rewards: $r_{\text{cnn}} = 2 \cdot p_{\text{cnn}}[y] - 1$ and $r_{\text{vit}} = 2 \cdot p_{\text{vit}}[y] - 1$, strictly bounded in $[-1, 1]$.
3. Incremental TD(0) sample-mean updates for $Q_{\text{cnn}}[s]$ and $Q_{\text{vit}}[s]$.
4. Track intermediate trajectory steps to plot the sigmoid saturation curves.

In [4]:
calibrator = RLFusionCalibrator()
val_probs_cnn = cnn_val["probs"].numpy()
val_probs_vit = vit_val["probs"].numpy()
val_labels = cnn_val["label"].numpy()
N_val = len(val_labels)

# History logging for trajectory analysis
val_history = []
trajectory_records = {c: [] for c in range(NUM_CLASSES)}

for i in range(N_val):
    y = int(val_labels[i])
    s = y   # In calibration, state is true ground-truth class

    r_cnn = 2.0 * float(val_probs_cnn[i, y]) - 1.0
    r_vit = 2.0 * float(val_probs_vit[i, y]) - 1.0

    # Pre-update weights for logging decision at step i
    w_c, w_v = calibrator.get_class_weights(s)
    fused_probs = w_c * val_probs_cnn[i] + w_v * val_probs_vit[i]
    fused_pred = int(np.argmax(fused_probs))

    val_history.append({
        "split": "val",
        "sample_idx": i,
        "state": s,
        "w_cnn": w_c,
        "w_vit": w_v,
        "r_cnn": r_cnn,
        "r_vit": r_vit,
        "cnn_correct": int(np.argmax(val_probs_cnn[i]) == y),
        "vit_correct": int(np.argmax(val_probs_vit[i]) == y),
        "fused_pred": fused_pred,
        "fused_correct": int(fused_pred == y),
    })

    # Perform incremental update
    calibrator.update(s, r_cnn, r_vit)

    # Record post-update weight trajectory for selected classes
    post_wc, _ = calibrator.get_class_weights(s)
    trajectory_records[s].append((calibrator.n[s], post_wc))

print(f"Finished calibration over {N_val} VAL samples.")
print("Calibrated sample counts per class:", calibrator.n)
for c in range(NUM_CLASSES):
    wc, wv = calibrator.get_class_weights(c)
    conf = calibrator.get_confidence(c)
    print(f"  Class {c:02d}: n={calibrator.n[c]:2d}, Q_cnn={calibrator.Q_cnn[c]:+.3f}, Q_vit={calibrator.Q_vit[c]:+.3f}, "
          f"Adv={calibrator.Q_cnn[c]-calibrator.Q_vit[c]:+.3f}, Conf={conf:.2f} -> w_cnn={wc:.3f}, w_vit={wv:.3f}")

w_cg, w_vg = calibrator.get_global_weights()
print(f"Global Pooled Weights (Ablation/Fallback): w_cnn={w_cg:.3f}, w_vit={w_vg:.3f}")

Finished calibration over 800 VAL samples.
Calibrated sample counts per class: [80 72 68 71 88 80 91 74 93 83]
  Class 00: n=80, Q_cnn=+0.388, Q_vit=+0.974, Adv=-0.586, Conf=1.00 -> w_cnn=0.263, w_vit=0.737
  Class 01: n=72, Q_cnn=+0.352, Q_vit=+0.963, Adv=-0.611, Conf=1.00 -> w_cnn=0.255, w_vit=0.745
  Class 02: n=68, Q_cnn=+0.199, Q_vit=+0.976, Adv=-0.776, Conf=1.00 -> w_cnn=0.207, w_vit=0.793
  Class 03: n=71, Q_cnn=+0.230, Q_vit=+0.980, Adv=-0.751, Conf=1.00 -> w_cnn=0.214, w_vit=0.786
  Class 04: n=88, Q_cnn=+0.344, Q_vit=+0.704, Adv=-0.359, Conf=1.00 -> w_cnn=0.345, w_vit=0.655
  Class 05: n=80, Q_cnn=+0.084, Q_vit=+0.859, Adv=-0.775, Conf=1.00 -> w_cnn=0.207, w_vit=0.793
  Class 06: n=91, Q_cnn=-0.046, Q_vit=+0.877, Adv=-0.923, Conf=1.00 -> w_cnn=0.173, w_vit=0.827
  Class 07: n=74, Q_cnn=-0.103, Q_vit=+0.938, Adv=-1.041, Conf=1.00 -> w_cnn=0.150, w_vit=0.850
  Class 08: n=93, Q_cnn=-0.292, Q_vit=+0.794, Adv=-1.086, Conf=1.00 -> w_cnn=0.142, w_vit=0.858
  Class 09: n=83, Q_cnn=+

## 6. Visualizations: Calibrated Weights & Saturation Curves

We produce two key visualizations required for the paper:
1. **Bar chart of $w_{\text{cnn}}[s]$ across all classes** (with the $0.5$ neutral baseline).
2. **Saturation curve over calibration step count** for sample classes showing how the sigmoid saturates smoothly.

In [5]:
classes = np.arange(NUM_CLASSES)
weights_cnn = [calibrator.get_class_weights(c)[0] for c in classes]
weights_vit = [calibrator.get_class_weights(c)[1] for c in classes]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Bar Chart across all classes
bars = ax1.bar(classes, weights_cnn, color="skyblue", edgecolor="black", alpha=0.85, label="CNN Weight ($w_{\\mathrm{cnn}}$)")
ax1.axhline(0.5, color="red", linestyle="--", linewidth=1.5, label="Neutral Prior (0.5)")
ax1.set_ylim(0.0, 1.0)
ax1.set_xlabel("Class ID (Context State $s$)", fontsize=11)
ax1.set_ylabel("Calibrated CNN Weight $w_{\\mathrm{cnn}}[s]$", fontsize=11)
ax1.set_title("Per-Class Calibrated Fusion Weights", fontsize=12, fontweight="bold")
ax1.set_xticks(classes)
ax1.legend(loc="upper right")
ax1.grid(True, linestyle=":", alpha=0.6)

# 2. Saturation Trajectory Curves for Selected Classes
selected_classes = [0, min(3, NUM_CLASSES-1), min(7, NUM_CLASSES-1)]
colors = ["#1f77b4", "#2ca02c", "#d62728"]
for idx, c in enumerate(selected_classes):
    if trajectory_records[c]:
        steps, w_vals = zip(*trajectory_records[c])
        ax2.plot(steps, w_vals, marker="o", markersize=3, label=f"Class {c}", color=colors[idx], linewidth=1.8)

ax2.axhline(0.5, color="gray", linestyle="--", alpha=0.7, label="Neutral Prior (0.5)")
ax2.set_ylim(0.0, 1.0)
ax2.set_xlabel("Per-Class Sample Count ($n[s]$)", fontsize=11)
ax2.set_ylabel("$w_{\\mathrm{cnn}}[s]$ Trajectory", fontsize=11)
ax2.set_title("Sigmoid Saturation Trajectory During Calibration", fontsize=12, fontweight="bold")
ax2.legend(loc="best")
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig("fusion_weight_calibration_plots.png", dpi=300)
plt.show()

<string>:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 7. Inference-Time Two-Pass Proxy State (TEST Split Evaluation)

At inference / deployment time, the true ground-truth label $y$ is unknown, so state cannot be $y$.
We implement the **two-pass proxy state estimate**:
- **Pass 1**: Preliminary consensus estimate $\hat{y}_0 = \arg\max(0.5 \cdot p_{\text{cnn}} + 0.5 \cdot p_{\text{vit}})$.
- **Pass 2**: Look up $w_{\text{cnn}}[\hat{y}_0]$ and $w_{\text{vit}}[\hat{y}_0]$. If calibration confidence for $\hat{y}_0$ was below $\text{CONF\_THRESHOLD} = 0.3$, fall back to the **global pooled weight** $(w_{\text{cnn}}^{\text{global}}, w_{\text{vit}}^{\text{global}})$.
- **Final Decision**: $\hat{y} = \arg\max(w_{\text{cnn}}[\hat{y}_0] \cdot p_{\text{cnn}} + w_{\text{vit}}[\hat{y}_0] \cdot p_{\text{vit}})$.

We also compute all baseline and ablation models on the test split for comparison.

In [6]:
test_probs_cnn = cnn_test["probs"].numpy()
test_probs_vit = vit_test["probs"].numpy()
test_labels = cnn_test["label"].numpy()
N_test = len(test_labels)

# Evaluate on TEST split
test_history = []
w_global_c, w_global_v = calibrator.get_global_weights()

fused_preds = []
naive_50_50_preds = []
global_ablation_preds = []

for i in range(N_test):
    p_c = test_probs_cnn[i]
    p_v = test_probs_vit[i]
    y_true = int(test_labels[i])

    # Baseline 1: Naive 50/50 average
    p_naive = 0.5 * p_c + 0.5 * p_v
    naive_pred = int(np.argmax(p_naive))
    naive_50_50_preds.append(naive_pred)

    # Baseline 2: Single Global Weight Ablation
    p_global = w_global_c * p_c + w_global_v * p_v
    global_pred = int(np.argmax(p_global))
    global_ablation_preds.append(global_pred)

    # Proposed Method: Two-Pass Contextual Proxy State
    # Pass 1: Unweighted rough guess
    y_hat0 = int(np.argmax(p_naive))

    # Pass 2: Look up contextual weights with confidence check
    conf = calibrator.get_confidence(y_hat0)
    if conf < CONF_THRESHOLD:
        # Fall back to global pooled weight if state had insufficient calibration data
        w_c, w_v = w_global_c, w_global_v
    else:
        w_c, w_v = calibrator.get_class_weights(y_hat0)

    p_fused = w_c * p_c + w_v * p_v
    final_pred = int(np.argmax(p_fused))
    fused_preds.append(final_pred)

    # Calculate per-sample rewards against true test label (for logging and analysis)
    r_cnn = 2.0 * float(p_c[y_true]) - 1.0
    r_vit = 2.0 * float(p_v[y_true]) - 1.0

    test_history.append({
        "split": "test",
        "sample_idx": i,
        "state": y_hat0,   # Proxy state in test
        "w_cnn": w_c,
        "w_vit": w_v,
        "r_cnn": r_cnn,
        "r_vit": r_vit,
        "cnn_correct": int(np.argmax(p_c) == y_true),
        "vit_correct": int(np.argmax(p_v) == y_true),
        "fused_pred": final_pred,
        "fused_correct": int(final_pred == y_true),
    })

# Combine (S, A, R) logs and export to CSV
all_records = val_history + test_history
sar_df = pd.DataFrame(all_records)
sar_df.to_csv("fusion_sar_log.csv", index=False)
print(f"Saved (S, A, R) logs to fusion_sar_log.csv with {len(sar_df)} total records.")

Saved (S, A, R) logs to fusion_sar_log.csv with 1600 total records.


## 8. Final Evaluation & Ablation Comparison Table (TEST Only)

The evaluation below is executed **once** on the TEST split.
We compare:
1. **CNN-only accuracy**
2. **ViT-only accuracy**
3. **Naive fixed 50/50 average accuracy**
4. **Per-class contextual fused accuracy** (Proposed Method)
5. **Single global-weight ablation** (Single state total: isolates the explicit contribution of per-class multi-state modeling vs. uniform adaptive weighting)

In [7]:
cnn_test_acc = float(cnn_test["correct"].float().mean().item()) * 100.0
vit_test_acc = float(vit_test["correct"].float().mean().item()) * 100.0
naive_50_50_acc = float(np.mean(np.array(naive_50_50_preds) == test_labels)) * 100.0
global_ablation_acc = float(np.mean(np.array(global_ablation_preds) == test_labels)) * 100.0
fused_acc = float(np.mean(np.array(fused_preds) == test_labels)) * 100.0

results_table = pd.DataFrame([
    {"Method / Configuration": "1. CNN Branch Only", "Test Accuracy (%)": f"{cnn_test_acc:.2f}%", "State Space": "None", "Adaptive": "No"},
    {"Method / Configuration": "2. ViT Branch Only", "Test Accuracy (%)": f"{vit_test_acc:.2f}%", "State Space": "None", "Adaptive": "No"},
    {"Method / Configuration": "3. Naive 50/50 Fixed Ensemble", "Test Accuracy (%)": f"{naive_50_50_acc:.2f}%", "State Space": "None", "Adaptive": "No"},
    {"Method / Configuration": "4. Single Global Weight (Ablation)", "Test Accuracy (%)": f"{global_ablation_acc:.2f}%", "State Space": "Single Global", "Adaptive": "Yes"},
    {"Method / Configuration": "5. Per-Class Contextual RL Fusion (Ours)", "Test Accuracy (%)": f"{fused_acc:.2f}%", "State Space": "Per-Class Proxy Context", "Adaptive": "Yes"},
])

print("=" * 80)
print("FINAL TEST SET EVALUATION & ABLATION COMPARISON")
print("=" * 80)
print(results_table.to_string(index=False))
print("=" * 80)

# Check advantage over baselines
best_single = max(cnn_test_acc, vit_test_acc)
print(f"Improvement over Best Single Branch: {fused_acc - best_single:+.2f}%")
print(f"Improvement over Naive 50/50 Ensemble:  {fused_acc - naive_50_50_acc:+.2f}%")
print(f"State Context Gain (Ours vs Global):  {fused_acc - global_ablation_acc:+.2f}%")

FINAL TEST SET EVALUATION & ABLATION COMPARISON
                  Method / Configuration Test Accuracy (%)             State Space Adaptive
                      1. CNN Branch Only            78.75%                    None       No
                      2. ViT Branch Only            97.00%                    None       No
           3. Naive 50/50 Fixed Ensemble            98.50%                    None       No
      4. Single Global Weight (Ablation)            97.62%           Single Global      Yes
5. Per-Class Contextual RL Fusion (Ours)            98.25% Per-Class Proxy Context      Yes
Improvement over Best Single Branch: +1.25%
Improvement over Naive 50/50 Ensemble:  -0.25%
State Context Gain (Ours vs Global):  +0.62%
